# Scraping Data from a Real Website + Pandas

In [1]:
import time
from bs4 import BeautifulSoup
import undetected_chromedriver as uc

options = uc.ChromeOptions()
driver = uc.Chrome(options=options)

url = "https://www.bps.go.id/id/statistics-table/1/MTAxMCMx/nilai-ekspor-menurut-negara-tujuan-utama--nilai-fob-juta-us----2000-2023.html"
driver.get(url)

time.sleep(10)

html = driver.page_source
driver.quit()

soup = BeautifulSoup(html, "html.parser")
tables = soup.find_all("table")

print(f": {len(tables)}")

: 1


In [10]:
print(soup.find('table').get('class'))

['xl654220']


In [11]:
table = soup.find('table', class_='xl654220')
print(table)

<table border="0" cellpadding="0" cellspacing="0" class="xl654220" style="border-collapse:collapse;width:1461pt;" width="1948">
<colgroup><col width="180"/>
<col span="26" width="68"/>
</colgroup><tbody><tr class="xl824220" style="height:15pt;">
<td class="xl834220" colspan="27" style="height:15pt;width:1461pt;" width="1948">Nilai Ekspor Menurut Negara/Wilayah/Entitas tertentu Tujuan
  Utama (Nilai FOB: juta US$),<span> </span>2000-2025</td>
</tr>
<tr style="height:15pt;">
<td class="xl674220" style="height:15pt;"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl654220"></td>
<td class="xl65

In [21]:
header_row = table.find_all('tr')[2]
headers = [' '.join(title.text.split()) for title in header_row.find_all(['td', 'th'])]
print(headers)

['Negara/Wilayah/Entitas tertentu', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [22]:
import pandas as pd

In [23]:
df = pd.DataFrame(columns = headers)
df

,Negara/Wilayah/Entitas tertentu,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025


In [24]:
from IPython.display import HTML, display

display(
    HTML("""
<style>
.jp-OutputArea-output pre {
    white-space: pre !important;
}
div.output_area pre {
    white-space: pre !important;
}
</style>
""")
)

In [25]:
column_data = table.find_all('tr')

for row in column_data:
    row_data = [
        ' '.join(td.text.split())
        for td in row.find_all('td')
        if td.text.strip() != ''
    ]

    if (
        len(row_data) > 1
        and row_data[0] != 'Negara/Wilayah/Entitas tertentu'
    ):
        print(row_data)

['ASEAN3)', '10.883,7', '9.507,1', '9.933,5', '10.725,4', '12.997,5', '15.824,9', '18.483,1', '22.292,1', '27.170,8', '24.624,0', '33.347,5', '42.098,9', '41.829,1', '40.630,0', '39.668,1', '33.577,0', '33.778,2', '39.266,4', '41.913,2', '41.464,5', '36.420,2', '48.115,2', '61.354,3', '53.086,4', '54.413,7', '58.030,7']
['Thailand', '1.026,5', '1.063,6', '1.227,4', '1.392,6', '1.976,2', '2.246,5', '2.701,5', '3.054,3', '3.661,3', '3.233,8', '4.566,6', '5.896,7', '6.635,1', '6.061,9', '5.783,1', '5.507,3', '5.394,0', '6.473,7', '6.820,9', '6.218,4', '5.110,3', '7.088,0', '8.196,8', '7.222,9', '7.707,2', '8.756,8']
['Singapura', '6.562,4', '5.363,9', '5.349,1', '5.399,7', '6.001,2', '7.836,6', '8.929,8', '10.501,6', '12.862,0', '10.262,7', '13.723,3', '18.443,9', '17.135,0', '16.686,3', '16.728,3', '12.632,6', '11.861,0', '12.724,9', '12.915,0', '12.916,7', '10.661,9', '11.635,8', '14.349,5', '12.607,6', '12.205,1', '13.438,2']
['Filipina', '819,5', '814,8', '778,2', '944,7', '1.237,6', 

In [50]:
import pandas as pd
import re

table = soup.find('table')
rows = table.find_all('tr')

data_parsed = []
benua_sekarang = None
grup_sekarang = None

BENUA_LIST = ['ASIA', 'AFRIKA', 'AUSTRALIA & OCEANIA', 'AMERIKA/AMERICA', 'EROPA']

for row in rows:
    cells = [' '.join(cell.text.split()) for cell in row.find_all(['td', 'th'])]
    cells = [c for c in cells if c != '']
    
    if not cells:
        continue
        
    teks_utama = cells[0]
    teks_bersih = re.sub(r'\d+\)', '', teks_utama).strip()
    
    if any(k in teks_utama for k in ['Nilai Eksport', 'Catatan:', 'Jumlah', 'TOTAL']) or teks_utama.startswith(('1)', '2)', '3)')):
        continue
    if teks_utama in ['Negara/Wilayah/Entitas tertentu', '2000']:
        continue
    if teks_bersih in BENUA_LIST:
        benua_sekarang = teks_bersih
        grup_sekarang = None
        if len(cells) > 1 and teks_bersih == 'AFRIKA':
            data_parsed.append([benua_sekarang, 'AFRIKA', 'AFRIKA'] + cells[1:])

    elif teks_bersih in ['ASEAN', 'NAFTA', 'Uni Eropa']:
        grup_sekarang = teks_bersih
    elif len(cells) > 1:
        if teks_bersih in ['Jepang', 'Hongkong', 'Korea Selatan', 'Taiwan', 'Tiongkok', 'Lainnya'] and benua_sekarang == 'ASIA':
            grup_sekarang = 'Asia Lainnya'

        elif 'Amerika Lainnya' in teks_bersih:
            grup_sekarang = 'Amerika Lainnya'

        elif 'Uni Eropa Lainnya' in teks_bersih:
            grup_sekarang = 'Uni Eropa'

        elif 'Eropa Lainnya' in teks_bersih:
            grup_sekarang = 'Eropa Lainnya'

        data_parsed.append([benua_sekarang, grup_sekarang, teks_bersih] + cells[1:])

header_row = table.find_all('tr')[2]
tahun_headers = [' '.join(th.text.split()) for th in header_row.find_all(['td', 'th']) if '20' in th.text]

kolom = ['Benua', 'Kawasan/Grup', 'Negara'] + tahun_headers
df = pd.DataFrame(data_parsed, columns=kolom)

pd.set_option('display.max_columns', None)
df

,Benua,Kawasan/Grup,Negara,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,ASIA,ASEAN,Thailand,"1.026,5","1.063,6","1.227,4","1.392,6","1.976,2","2.246,5","2.701,5","3.054,3","3.661,3","3.233,8","4.566,6","5.896,7","6.635,1","6.061,9","5.783,1","5.507,3","5.394,0","6.473,7","6.820,9","6.218,4","5.110,3","7.088,0","8.196,8","7.222,9","7.707,2","8.756,8"
1,ASIA,ASEAN,Singapura,"6.562,4","5.363,9","5.349,1","5.399,7","6.001,2","7.836,6","8.929,8","10.501,6","12.862,0","10.262,7","13.723,3","18.443,9","17.135,0","16.686,3","16.728,3","12.632,6","11.861,0","12.724,9","12.915,0","12.916,7","10.661,9","11.635,8","14.349,5","12.607,6","12.205,1","13.438,2"
2,ASIA,ASEAN,Filipina,"819,5","814,8","778,2","944,7","1.237,6","1.419,1","1.405,7","1.853,7","2.053,6","2.405,9","3.180,7","3.699,0","3.707,6","3.817,0","3.887,8","3.921,7","5.270,9","6.629,6","6.832,2","6.770,1","5.900,7","8.604,3","12.905,1","11.040,1","10.748,4","10.220,7"
3,ASIA,ASEAN,Malaysia,"1.971,8","1.778,6","2.029,9","2.363,9","3.016,0","3.431,3","4.110,8","5.096,1","6.432,5","6.811,8","9.362,3","10.995,8","11.278,3","10.666,6","9.730,0","7.630,9","7.069,5","8.441,1","9.271,6","8.801,8","8.098,8","12.064,7","15.672,6","12.804,4","12.548,8","13.061,0"
4,ASIA,ASEAN,Myanmar,"64,8","69,0","54,4","45,7","60,3","78,0","137,7","262,4","250,8","174,8","284,2","359,5","401,6","556,4","566,9","615,7","615,7","827,5","897,6","875,6","1.031,9","1.122,1","964,6","796,6","620,1","787,8"
5,ASIA,ASEAN,Kamboja,"51,7","72,1","68,8","79,9","71,8","93,9","103,6","121,8","174,0","201,2","217,7","259,5","292,2","312,4","415,8","429,7","426,9","513,9","525,6","618,5","541,3","531,2","725,6","856,1","833,3","869,3"
6,ASIA,ASEAN,Brunei Darussalam,"25,5","21,6","32,1","30,4","31,8","39,3","37,6","43,4","59,7","74,9","61,0","81,7","81,8","122,7","100,3","91,2","88,7","64,5","61,2","103,2","129,0","211,4","240,1","197,2","227,8","187,5"
7,ASIA,ASEAN,Laos,"0,9","1,4","0,7","0,5","1,6","1,8","4,3","3,7","4,0","4,7","5,5","8,6","23,8","5,8","4,6","7,7","5,9","4,2","7,3","6,8","5,1","7,6","26,6","16,7","15,0","13,3"
8,ASIA,ASEAN,Vietnam,"360,6","322,1","392,9","468,1","601,0","678,4","1.052,0","1.355,1","1.672,9","1.454,2","1.946,2","2.354,2","2.273,7","2.400,9","2.451,3","2.740,2","3.045,6","3.586,9","4.581,9","5.153,4","4.941,4","6.850,1","8.273,3","7.544,9","9.508,0","10.629,1"
9,ASIA,ASEAN,Timor Leste,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,"67,0"


In [51]:
df.to_csv(
    r'E:\Python project\web scraping with jupyter notebook\data_expor_BPS.csv',
    index=False)